In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
import os
os.environ['KAGGLE_CONFIG_DIR'] = "/content/drive/MyDrive/kaggle"

In [ ]:
!pip install -q kaggle

!kaggle datasets download aminasalamt/social-media-dataset-2025

In [ ]:
import zipfile

with zipfile.ZipFile("social-media-dataset-2025.zip", "r") as z:
    z.extractall("/content/data")

In [ ]:
import os

os.listdir("data")

In [ ]:
import pandas as pd

df = pd.read_csv("data/Students Social Media Addiction.csv")
df.head()

In [ ]:
df.info()
df.describe()

In [ ]:
df.columns = df.columns.str.strip()

In [ ]:
import pandas as pd

# Check missing values (FULL dataset)
print("MISSING VALUES:")
print(df.isnull().sum())

# Check duplicates (FULL dataset)
print("\nDUPLICATE ROWS:")
print(df.duplicated().sum())

In [ ]:
print("\nDATA TYPES:\n", df.dtypes)

df['Avg_Daily_Usage_Hours'] = pd.to_numeric(df['Avg_Daily_Usage_Hours'], errors='coerce')
df['Sleep_Hours_Per_Night'] = pd.to_numeric(df['Sleep_Hours_Per_Night'], errors='coerce')
df['Mental_Health_Score'] = pd.to_numeric(df['Mental_Health_Score'], errors='coerce')

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(df['Avg_Daily_Usage_Hours'],
            df['Sleep_Hours_Per_Night'],
            alpha=0.5)

plt.xlabel("Daily Social Media Usage (Hours)")
plt.ylabel("Sleep Hours Per Night")
plt.title("Social Media Usage vs Sleep (Full Dataset)")
plt.show()

In [ ]:
import seaborn as sns

sns.boxplot(x='Affects_Academic_Performance',
            y='Avg_Daily_Usage_Hours',
            data=df)

plt.title("Usage vs Academic Performance (All Levels)")

In [ ]:
import seaborn as sns

sns.regplot(x='Avg_Daily_Usage_Hours',
            y='Mental_Health_Score',
            data=df)

plt.title("Usage vs Mental Health (Full Dataset)")

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


df_model = df.copy()

# Clean column names
df_model.columns = df_model.columns.str.strip()


le = LabelEncoder()
df_model['Affects_Academic_Performance'] = le.fit_transform(
    df_model['Affects_Academic_Performance']
)

# Add Academic_Level as feature
df_model = pd.get_dummies(df_model, columns=['Academic_Level'], drop_first=True)

# Features
X = df_model[
    ['Avg_Daily_Usage_Hours',
     'Sleep_Hours_Per_Night',
     'Mental_Health_Score',
     'Age'] +
    [col for col in df_model.columns if col.startswith('Academic_Level_')]
]

y = df_model['Affects_Academic_Performance']

# Stratified Cross Validation
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000)
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy')

print("Scores:", scores)
print("Average accuracy:", scores.mean())


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(6,4))

sns.heatmap(df[['Avg_Daily_Usage_Hours',
                'Sleep_Hours_Per_Night',
                'Mental_Health_Score',
                'Age']].corr(),
            annot=True, cmap='coolwarm')

plt.title("Correlation Heatmap (Full Dataset)")

In [ ]:
df[['Avg_Daily_Usage_Hours',
    'Sleep_Hours_Per_Night',
    'Mental_Health_Score',
    'Age']].corr()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

# Example: predict whether academic performance is affected
df = df.dropna()

X = df[['Avg_Daily_Usage_Hours', 'Sleep_Hours_Per_Night', 'Mental_Health_Score', 'Age']]
y = df['Affects_Academic_Performance']

# Convert categorical target to numbers if needed
y = y.astype('category').cat.codes

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train model
model = LogisticRegression()
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, y_pred)

sns.heatmap(cm, annot=True, fmt='d')
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
sns.pairplot(df, hue='Affects_Academic_Performance')

In [ ]:
sns.boxplot(x='Academic_Level', y='Avg_Daily_Usage_Hours', data=df)
plt.xticks(rotation=45)
plt.title("Usage by Academic Level")
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

df_model = df.copy()

le = LabelEncoder()
df_model['Affects_Academic_Performance'] = le.fit_transform(
    df_model['Affects_Academic_Performance']
)

X = df_model[['Avg_Daily_Usage_Hours',
              'Sleep_Hours_Per_Night',
              'Mental_Health_Score',
              'Age']]

y = df_model['Affects_Academic_Performance']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

model = LogisticRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

y_pred_knn = knn.predict(X_test)

print("KNN Accuracy:", accuracy_score(y_test, y_pred_knn))

In [ ]:
for k in range(1, 11):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    print("K =", k, "Accuracy =", accuracy_score(y_test, knn.predict(X_test)))

In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(model, X, y, cv=5)

print("Cross-validation scores:", scores)
print("Average score:", scores.mean())

In [ ]:
X1 = df_model[['Avg_Daily_Usage_Hours']]

X2 = df_model[['Avg_Daily_Usage_Hours', 'Sleep_Hours_Per_Night']]

X3 = df_model[['Avg_Daily_Usage_Hours',
               'Sleep_Hours_Per_Night',
               'Mental_Health_Score']]